## 22_functional_regions
  Community detection (Leiden, Louvain, greedy modularity) on the observed CZ obec and PL gmina
  commuting networks, compared with NUTS3 (NMI, ARI, purity).
  Output: data/processed/22_functional_regions/.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# ---- Config (all names, paths and thresholds live here) ----
NOTEBOOK = "22_functional_regions"
CZ_RAW_NAME = "dojizdka_obce.csv"
CZ_URL = ("https://csu.gov.cz/docs/107508/4dbdab3b-905c-deff-4cfa-e4828a6fa2de/"
          "dojizdka_obce.csv")
CZ_MIN_BYTES = 10_000_000          # the file is ~35 MB, anything smaller is broken
CZ_MANUAL_HINT = ("Source: CSU SLDB 2021, 'Dojizdka mezi obcemi, vcetne denni dojizdky' "
                  "(data.gov.cz).")
# -------------------------------------------------------------

# Project root: the notebook runs from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from mobility.data_io import ensure_file

RAW_CZ = ROOT / "data" / "raw" / "czso"
OUT_DIR = ROOT / "data" / "processed" / NOTEBOOK
RAW_CZ.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Skips if the file exists, searches under data/, downloads only if missing
cz_csv = ensure_file(
    RAW_CZ / CZ_RAW_NAME,
    url=CZ_URL,
    base=ROOT,
    min_bytes=CZ_MIN_BYTES,
    search_root=ROOT / "data",
    manual_hint=CZ_MANUAL_HINT,
)

print("Used file:", cz_csv.relative_to(ROOT).as_posix())
print("Output folder:", OUT_DIR.relative_to(ROOT).as_posix())

In [ ]:
import csv

# ---- Config (names, reference sums and filters live here) ----
CZ_OUT_NAME = "cz_od_work_obec_rebuilt.parquet"
COL_TYPE = "lokalizace"
COL_FLOW = "dojizdka_prace"
COL_ORIG = "op_obec_kod"
COL_DEST = "doj_obec_kod"
TYPE_INTER = "1_meziobecni"
PSEUDO_CODES = {"888888", "999999"}        # non-municipality destination codes
# Columns kept in the work table (same list as the archived 08_cz_od_audit)
KEEP_COLS = [COL_TYPE, "op_lau1_kod", "op_orp_kod", COL_ORIG, "op_obec",
             "doj_lau1_kod", "doj_orp_kod", COL_DEST, "doj_obec", COL_FLOW]
# Totals published by CSU (dojizdka_prace), used as a check, not as input
CSU_REF = {
    "0_na_adrese_OP": 1_011_538,
    "1_meziobecni": 2_029_425,
    "2_zahranici": 74_198,
    "3_v_ramci_obce": 1_539_021,
}
ENCODINGS = ("utf-8-sig", "cp1250", "utf-8")
# ---------------------------------------------------------------

def sniff_csv(path, encodings=ENCODINGS):
    """Guess encoding and delimiter from the header line; data is not modified."""
    for enc in encodings:
        try:
            with open(path, encoding=enc, newline="") as f:
                head = f.read(50_000)
            dialect = csv.Sniffer().sniff(head.split("\n", 1)[0], delimiters=",;\t|")
            return enc, dialect.delimiter
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError("Could not detect encoding/delimiter")

cz_out = OUT_DIR / CZ_OUT_NAME

enc, sep = sniff_csv(cz_csv)
od_raw = pd.read_csv(cz_csv, dtype=str, encoding=enc, sep=sep)
od_raw[COL_FLOW] = pd.to_numeric(od_raw[COL_FLOW], errors="raise").astype("int64")
print("raw shape:", od_raw.shape)

# 1) Check the category totals against the CSU publication
sums = od_raw.groupby(COL_TYPE)[COL_FLOW].sum()
for key, ref in CSU_REF.items():
    status = "OK" if sums[key] == ref else "MISMATCH"
    print(f"{key:22s} data: {sums[key]:>10,} | CSU: {ref:>10,} | {status}")

# 2) Clean work OD: inter-municipal, positive flows only, kept columns only
inter = od_raw[od_raw[COL_TYPE] == TYPE_INTER]
cz_work = inter.loc[inter[COL_FLOW] > 0, KEEP_COLS].reset_index(drop=True)

assert (cz_work[COL_ORIG] != cz_work[COL_DEST]).all(), "diagonal rows present"
assert not cz_work[COL_DEST].isin(PSEUDO_CODES).any(), "pseudo destination codes present"
assert not cz_work.duplicated([COL_ORIG, COL_DEST]).any(), "duplicate pairs present"
assert cz_work[COL_FLOW].sum() == CSU_REF[TYPE_INTER], "N_c differs from the CSU total"

print(f"cz_work: {len(cz_work):,} rows | N_c = {int(cz_work[COL_FLOW].sum()):,}")

# 3) Save; skip only if the existing file already has the expected columns
if cz_out.exists() and list(pd.read_parquet(cz_out).columns) == KEEP_COLS:
    print("[exists with expected columns, skipped]", cz_out.relative_to(ROOT).as_posix())
else:
    cz_work.to_parquet(cz_out, index=False)
    print("[saved]", cz_out.relative_to(ROOT).as_posix())

In [ ]:
# ---- Config ----
CZ_EXISTING = ROOT / "data" / "interim" / "cz_od_work_obec.parquet"
CZ_COMPARE_NAME = "cz_rebuild_comparison.csv"
KEY_COLS = [COL_ORIG, COL_DEST]
# ----------------

cz_old = pd.read_parquet(CZ_EXISTING)
cz_new = pd.read_parquet(cz_out)

# 1) Same columns (as a set and in the same order)
same_cols_set = set(cz_old.columns) == set(cz_new.columns)
same_cols_order = list(cz_old.columns) == list(cz_new.columns)

# 2) Sort both by the key pair so row order does not matter
old_s = cz_old.sort_values(KEY_COLS).reset_index(drop=True)
new_s = cz_new[cz_old.columns if same_cols_set else cz_new.columns] \
    .sort_values(KEY_COLS).reset_index(drop=True)

same_rows = len(old_s) == len(new_s)
identical = same_cols_set and same_rows and old_s.equals(new_s)

# 3) If not identical, count differing pairs by outer merge on the keys
n_only_old = n_only_new = n_flow_diff = None
if not identical:
    merged = old_s.merge(new_s, on=KEY_COLS, how="outer",
                         suffixes=("_old", "_new"), indicator=True)
    n_only_old = int((merged["_merge"] == "left_only").sum())
    n_only_new = int((merged["_merge"] == "right_only").sum())
    both = merged[merged["_merge"] == "both"]
    n_flow_diff = int((both[COL_FLOW + "_old"] != both[COL_FLOW + "_new"]).sum())

summary = pd.Series({
    "rows_existing": len(cz_old),
    "rows_rebuilt": len(cz_new),
    "same_columns_as_set": same_cols_set,
    "same_columns_in_order": same_cols_order,
    "flow_sum_existing": int(cz_old[COL_FLOW].sum()),
    "flow_sum_rebuilt": int(cz_new[COL_FLOW].sum()),
    "identical_after_sort": identical,
    "pairs_only_in_existing": n_only_old,
    "pairs_only_in_rebuilt": n_only_new,
    "pairs_with_different_flow": n_flow_diff,
}, name="value")

print("AZONOS" if identical else "ELTÉRÉS")
print(summary.to_string())

# Save the small summary so it can be read without re-running the notebook
cmp_out = OUT_DIR / CZ_COMPARE_NAME
summary.to_csv(cmp_out)
print("[saved]", cmp_out.relative_to(ROOT).as_posix())

In [ ]:
# ---- Config (names, sheet names and reference values live here) ----
PL_XLSX_NAME = "macierz_przeplywow_ludnosci_zwiazanych_z_zatrudnieniem.xlsx"
GUS_URL = ("https://stat.gov.pl/download/gfx/portalinformacyjny/pl/defaultaktualnosci/"
           "6536/9/2/1/" + PL_XLSX_NAME)
PL_URL = "https://web.archive.org/web/2025id_/" + GUS_URL   # archive copy (server cannot reach stat.gov.pl)
PL_MIN_BYTES = 1_000_000
PL_MANUAL_HINT = ("Download the GUS file (stat.gov.pl or its web.archive.org copy) "
                  "and put it into data/raw/gus/")
SHEET_MATRIX = "Macierz przepływów"
SHEET_OUT = "Wyjeżdżający z gmin"
SHEET_IN = "Przyjeżdżający do gmin"
RX_CODE = r"\d{7}"                 # valid 7-digit unit code
KEY_LEN = 6                        # 6-digit gmina key (parts of a gmina share it)
GUS_REF_TOTAL = 5_182_600          # commuters published by GUS (5182.6 thousand)
PL_OUT_NAME = "pl_od_work_gmina_rebuilt.parquet"
PL_COLS = ["p6_r", "p6_w", "flow"]
# ---------------------------------------------------------------------

RAW_PL = ROOT / "data" / "raw" / "gus"
RAW_PL.mkdir(parents=True, exist_ok=True)

# Skips if the file exists, downloads only if missing, prints a manual hint on failure
pl_xlsx = ensure_file(
    RAW_PL / PL_XLSX_NAME,
    url=PL_URL,
    base=ROOT,
    min_bytes=PL_MIN_BYTES,
    search_root=ROOT / "data",
    manual_hint=PL_MANUAL_HINT,
)
print("Used file:", pl_xlsx.relative_to(ROOT).as_posix())

xl = pd.ExcelFile(pl_xlsx)

# 1) Matrix sheet: all text, keep rows with valid 7-digit codes on both sides
mat = xl.parse(SHEET_MATRIX, dtype=str)
RC, RN, WC, WN, FL = list(mat.columns)[:5]
good = mat[RC].str.fullmatch(RX_CODE, na=False) & mat[WC].str.fullmatch(RX_CODE, na=False)
data = mat[good].copy()
data["flow"] = pd.to_numeric(data[FL], errors="raise")
assert (data["flow"] > 0).all() and (data["flow"] % 1 == 0).all(), "non-positive or non-integer flow"
print(f"matrix rows: {len(mat):,} | valid: {len(data):,} | flow sum: {int(data['flow'].sum()):,}")

# 2) Control sheets: totals must match the figure published by GUS
for name, sheet in [("out", SHEET_OUT), ("in", SHEET_IN)]:
    ctl = xl.parse(sheet, dtype=str)
    code_col, val_col = list(ctl.columns)[0], list(ctl.columns)[2]
    ctl = ctl[ctl[code_col].str.fullmatch(RX_CODE, na=False)]
    total = int(pd.to_numeric(ctl[val_col]).sum())
    print(f"control sheet {name}: {total:,} | GUS: {GUS_REF_TOTAL:,} | diff: {total - GUS_REF_TOTAL:,}")

# 3) Gmina-level OD: 6-digit keys, drop rows inside one gmina, sum flows per pair
data["p6_r"] = data[RC].str[:KEY_LEN]
data["p6_w"] = data[WC].str[:KEY_LEN]
inter = data[data["p6_r"] != data["p6_w"]]
pl_work = inter.groupby(["p6_r", "p6_w"], as_index=False)["flow"].sum()
pl_work["flow"] = pl_work["flow"].astype("int64")
print(f"gmina pairs: {len(pl_work):,} | flow sum: {int(pl_work['flow'].sum()):,} "
      f"| smallest flow: {int(pl_work['flow'].min())}")

# 4) Save; skip only if the existing file already has the expected columns
pl_out = OUT_DIR / PL_OUT_NAME
if pl_out.exists() and list(pd.read_parquet(pl_out).columns) == PL_COLS:
    print("[exists with expected columns, skipped]", pl_out.relative_to(ROOT).as_posix())
else:
    pl_work.to_parquet(pl_out, index=False)
    print("[saved]", pl_out.relative_to(ROOT).as_posix())

In [ ]:
# ---- Config ----
PL_EXISTING = ROOT / "data" / "interim" / "pl_od_work_gmina.parquet"
PL_COMPARE_NAME = "pl_rebuild_comparison.csv"
PL_KEY_COLS = ["p6_r", "p6_w"]
PL_FLOW_COL = "flow"
# ----------------

pl_old = pd.read_parquet(PL_EXISTING)
pl_new = pd.read_parquet(pl_out)

# 1) Same columns (as a set and in the same order)
same_cols_set = set(pl_old.columns) == set(pl_new.columns)
same_cols_order = list(pl_old.columns) == list(pl_new.columns)

# 2) Sort both by the key pair so row order does not matter
old_s = pl_old.sort_values(PL_KEY_COLS).reset_index(drop=True)
new_s = pl_new[pl_old.columns if same_cols_set else pl_new.columns] \
    .sort_values(PL_KEY_COLS).reset_index(drop=True)

same_rows = len(old_s) == len(new_s)
identical = same_cols_set and same_rows and old_s.equals(new_s)

# 3) If not identical, count differing pairs by outer merge on the keys
n_only_old = n_only_new = n_flow_diff = None
if not identical:
    merged = old_s.merge(new_s, on=PL_KEY_COLS, how="outer",
                         suffixes=("_old", "_new"), indicator=True)
    n_only_old = int((merged["_merge"] == "left_only").sum())
    n_only_new = int((merged["_merge"] == "right_only").sum())
    both = merged[merged["_merge"] == "both"]
    n_flow_diff = int((both[PL_FLOW_COL + "_old"] != both[PL_FLOW_COL + "_new"]).sum())

summary = pd.Series({
    "rows_existing": len(pl_old),
    "rows_rebuilt": len(pl_new),
    "same_columns_as_set": same_cols_set,
    "same_columns_in_order": same_cols_order,
    "dtypes_existing": ", ".join(f"{c}:{t}" for c, t in pl_old.dtypes.astype(str).items()),
    "dtypes_rebuilt": ", ".join(f"{c}:{t}" for c, t in pl_new.dtypes.astype(str).items()),
    "flow_sum_existing": int(pl_old[PL_FLOW_COL].sum()),
    "flow_sum_rebuilt": int(pl_new[PL_FLOW_COL].sum()),
    "identical_after_sort": identical,
    "pairs_only_in_existing": n_only_old,
    "pairs_only_in_rebuilt": n_only_new,
    "pairs_with_different_flow": n_flow_diff,
}, name="value")

print("AZONOS" if identical else "ELTÉRÉS")
print(summary.to_string())

# Save the small summary so it can be read without re-running the notebook
cmp_out_pl = OUT_DIR / PL_COMPARE_NAME
summary.to_csv(cmp_out_pl)
print("[saved]", cmp_out_pl.relative_to(ROOT).as_posix())

In [ ]:
import networkx as nx
import numpy as np

# ---- Config (one block: tables, column names, output names) ----
GRAPH_SUMMARY_NAME = "graph_summary.csv"
LEVELS = {
    "CZ": {"path": cz_out, "orig": COL_ORIG, "dest": COL_DEST, "flow": COL_FLOW,
           "edges_name": "cz_edges_undirected.parquet",
           "graphml_name": "cz_graph.graphml"},
    "PL": {"path": pl_out, "orig": "p6_r", "dest": "p6_w", "flow": "flow",
           "edges_name": "pl_edges_undirected.parquet",
           "graphml_name": "pl_graph.graphml"},
}
# -----------------------------------------------------------------

graphs = {}
summary_rows = []

for level, cfg in LEVELS.items():
    od = pd.read_parquet(cfg["path"], columns=[cfg["orig"], cfg["dest"], cfg["flow"]])
    edges_path = OUT_DIR / cfg["edges_name"]
    gml_path = OUT_DIR / cfg["graphml_name"]

    # Load the edge checkpoint if it exists, otherwise build and save it
    if edges_path.exists():
        edges = pd.read_parquet(edges_path)
        print(f"[loaded] {edges_path.relative_to(ROOT).as_posix()}")
    else:
        # Symmetrize: order each pair so that node_a <= node_b, then add the two directions
        swap = od[cfg["orig"]] > od[cfg["dest"]]
        node_a = np.where(swap, od[cfg["dest"]], od[cfg["orig"]])
        node_b = np.where(swap, od[cfg["orig"]], od[cfg["dest"]])
        edges = (od.assign(node_a=node_a, node_b=node_b)
                   .groupby(["node_a", "node_b"], as_index=False)[cfg["flow"]].sum()
                   .rename(columns={cfg["flow"]: "weight"}))
        edges.to_parquet(edges_path, index=False)
        print(f"[saved] {edges_path.relative_to(ROOT).as_posix()}")

    # Total weight must equal the total directed flow (checked on loaded files too)
    assert edges["weight"].sum() == od[cfg["flow"]].sum(), f"{level}: weight sum differs"

    G = nx.Graph()
    G.add_weighted_edges_from(edges[["node_a", "node_b", "weight"]].itertuples(index=False, name=None))
    graphs[level] = G

    # Save the finished graph as GraphML (skip if it already exists)
    if gml_path.exists():
        print(f"[exists, skipped] {gml_path.relative_to(ROOT).as_posix()}")
    else:
        nx.write_graphml(G, gml_path)
        print(f"[saved] {gml_path.relative_to(ROOT).as_posix()} "
              f"({gml_path.stat().st_size / 1e6:.1f} MB)")

    comps = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    summary_rows.append({
        "level": level,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "total_weight": int(edges["weight"].sum()),
        "directed_pairs": len(od),
        "components": len(comps),
        "largest_component_nodes": comps[0],
    })

graph_summary = pd.DataFrame(summary_rows)
print(graph_summary.to_string(index=False))

summary_path = OUT_DIR / GRAPH_SUMMARY_NAME
graph_summary.to_csv(summary_path, index=False)
print("[saved]", summary_path.relative_to(ROOT).as_posix())

In [ ]:
import random
import igraph as ig

# ---- Config ----
SEED = 42
WEIGHT_ATTR = "weight"
IGRAPH_METHODS = ["leiden", "louvain"]
COMMUNITY_SUMMARY_NAME = "community_summary.csv"
# ----------------

def to_igraph(G):
    """Convert the weighted networkx graph to igraph; node codes become vertex names."""
    triples = ((u, v, d[WEIGHT_ATTR]) for u, v, d in G.edges(data=True))
    return ig.Graph.TupleList(triples, weights=True)

def run_method(g, method):
    """Run one igraph community method with a fixed seed."""
    random.seed(SEED)                      # igraph draws from Python's random module
    if method == "leiden":
        # modularity objective, iterate until the partition no longer changes
        return g.community_leiden(objective_function="modularity",
                                  weights=WEIGHT_ATTR, n_iterations=-1)
    if method == "louvain":
        return g.community_multilevel(weights=WEIGHT_ATTR)
    raise ValueError(method)

summary_rows = []
for level, G in graphs.items():
    g = to_igraph(G)
    names = g.vs["name"]

    for method in IGRAPH_METHODS:
        members_path = OUT_DIR / f"{level.lower()}_communities_{method}.parquet"

        # Checkpoint: load the saved partition if it exists, otherwise compute and save
        if members_path.exists():
            members = pd.read_parquet(members_path)
            print(f"[loaded] {members_path.relative_to(ROOT).as_posix()}")
        else:
            clustering = run_method(g, method)
            members = pd.DataFrame({"node": names, "community": clustering.membership})
            members.to_parquet(members_path, index=False)
            print(f"[saved] {members_path.relative_to(ROOT).as_posix()}")

        # Newman modularity from networkx, on the same weighted graph for every method
        groups = [set(grp["node"]) for _, grp in members.groupby("community")]
        assert sum(len(s) for s in groups) == G.number_of_nodes(), "not a partition of all nodes"
        q = nx.community.modularity(G, groups, weight=WEIGHT_ATTR)

        sizes = members["community"].value_counts()
        summary_rows.append({
            "level": level,
            "method": method,
            "communities": len(groups),
            "modularity_newman": round(q, 4),
            "largest_community_nodes": int(sizes.max()),
            "smallest_community_nodes": int(sizes.min()),
            "seed": SEED,
        })

community_summary = pd.DataFrame(summary_rows)
print(community_summary.to_string(index=False))

summary_path = OUT_DIR / COMMUNITY_SUMMARY_NAME
community_summary.to_csv(summary_path, index=False)
print("[saved]", summary_path.relative_to(ROOT).as_posix())

In [ ]:
import time

# ---- Config ----
CNM_METHOD = "cnm"
# ----------------

cnm_rows = []
for level, G in graphs.items():
    members_path = OUT_DIR / f"{level.lower()}_communities_{CNM_METHOD}.parquet"

    # Checkpoint: load the saved partition if it exists, otherwise compute and save
    if members_path.exists():
        members = pd.read_parquet(members_path)
        print(f"[loaded] {members_path.relative_to(ROOT).as_posix()}")
    else:
        t0 = time.time()
        parts = nx.community.greedy_modularity_communities(G, weight=WEIGHT_ATTR)
        rows = [(node, i) for i, part in enumerate(parts) for node in part]
        members = pd.DataFrame(rows, columns=["node", "community"])
        members.to_parquet(members_path, index=False)
        print(f"[saved] {members_path.relative_to(ROOT).as_posix()} "
              f"({time.time() - t0:.0f} s)")

    # Newman modularity with the same networkx function as for the other methods
    groups = [set(grp["node"]) for _, grp in members.groupby("community")]
    assert sum(len(s) for s in groups) == G.number_of_nodes(), "not a partition of all nodes"
    q = nx.community.modularity(G, groups, weight=WEIGHT_ATTR)

    sizes = members["community"].value_counts()
    cnm_rows.append({
        "level": level,
        "method": CNM_METHOD,
        "communities": len(groups),
        "modularity_newman": round(q, 4),
        "largest_community_nodes": int(sizes.max()),
        "smallest_community_nodes": int(sizes.min()),
        "seed": None,
    })

# Keep the earlier rows, replace any previous cnm rows, and save the updated summary
community_summary = pd.concat(
    [community_summary[community_summary["method"] != CNM_METHOD], pd.DataFrame(cnm_rows)],
    ignore_index=True,
).sort_values(["level", "method"]).reset_index(drop=True)
print(community_summary.to_string(index=False))

community_summary.to_csv(summary_path, index=False)
print("[saved]", summary_path.relative_to(ROOT).as_posix())

In [ ]:
# ---- Config (all names, slices and expected values live here) ----
EUROSTAT_NAME = "EU-27-LAU-2021-NUTS-2021.xlsx"
EUROSTAT_URL = "https://ec.europa.eu/eurostat/documents/345175/501971/" + EUROSTAT_NAME
EUROSTAT_MIN_BYTES = 100_000
EUROSTAT_HINT = ("Source: Eurostat, 'LAU - NUTS 2021' table; put the file into "
                 "data/raw/eurostat/")
SHEET_COUNTRY = "PL"
NUTS_COLS = {"nuts3": "NUTS 3 CODE", "lau": "LAU CODE"}   # original Eurostat column names
P6_SLICES = ((4, 6), (9, 13))      # gmina key = LAU CODE[4:6] + LAU CODE[9:13] (as in archived 14)
LAU_ID_LEN = 13
POWIAT_PREFIX_LEN = 4              # powiat code = first 4 characters of the gmina key
OUT_COLS = {"gmina": "gmina_code", "lau": "lau_id", "powiat": "powiat_code", "nuts3": "nuts3_code"}
MAP_OUT_NAME = "pl_gmina_nuts3_rebuilt.csv"
MAP_EXISTING = ROOT / "data" / "interim" / "pl_gmina_nuts3.csv"
MAP_COMPARE_NAME = "pl_gmina_nuts3_comparison.csv"
# ------------------------------------------------------------------

RAW_EUROSTAT = ROOT / "data" / "raw" / "eurostat"
RAW_EUROSTAT.mkdir(parents=True, exist_ok=True)

# Skips if the file exists, downloads only if missing, prints a manual hint on failure
eurostat_xlsx = ensure_file(
    RAW_EUROSTAT / EUROSTAT_NAME,
    url=EUROSTAT_URL,
    base=ROOT,
    min_bytes=EUROSTAT_MIN_BYTES,
    search_root=ROOT / "data",
    manual_hint=EUROSTAT_HINT,
)
print("Used file:", eurostat_xlsx.relative_to(ROOT).as_posix())

# 1) Read the PL sheet (xl_lau, so the PL commuting workbook object xl stays untouched)
xl_lau = pd.ExcelFile(eurostat_xlsx)
if SHEET_COUNTRY not in xl_lau.sheet_names:
    raise ValueError(f"Sheet {SHEET_COUNTRY} not found")
raw_lau = xl_lau.parse(SHEET_COUNTRY, dtype=str)

lau_c, nuts_c = NUTS_COLS["lau"], NUTS_COLS["nuts3"]
m = raw_lau[[lau_c, nuts_c]].astype(str).apply(lambda s: s.str.strip())
assert m[lau_c].str.fullmatch(rf"\d{{{LAU_ID_LEN}}}").all(), "not every LAU CODE has 13 digits"
assert not m[lau_c].duplicated().any(), "duplicate LAU CODE"

# 2) Gmina key and powiat code
m[OUT_COLS["gmina"]] = m[lau_c].str[slice(*P6_SLICES[0])] + m[lau_c].str[slice(*P6_SLICES[1])]
m[OUT_COLS["powiat"]] = m[OUT_COLS["gmina"]].str[:POWIAT_PREFIX_LEN]
assert not m[OUT_COLS["gmina"]].duplicated().any(), "duplicate gmina key"
print("Eurostat PL rows:", len(m), "| NUTS3 units:", m[nuts_c].nunique())

# 3) Every node of the PL graph must have a NUTS3 code
nodes = set(graphs["PL"].nodes)
keys = set(m[OUT_COLS["gmina"]])
print(f"graph nodes: {len(nodes)} | without NUTS3: {len(nodes - keys)} "
      f"| in the table only: {len(keys - nodes)}")
assert not (nodes - keys), "graph nodes without a NUTS3 code"

# 4) A powiat must fall into exactly one NUTS3 region
per_powiat = m.groupby(OUT_COLS["powiat"])[nuts_c].nunique()
print(f"powiats: {len(per_powiat)} | powiats in more than one NUTS3: {int((per_powiat > 1).sum())}")
assert (per_powiat == 1).all(), "a powiat falls into several NUTS3 regions"

# 5) Output table with the same columns as the archived file
pl_map = m.rename(columns={lau_c: OUT_COLS["lau"], nuts_c: OUT_COLS["nuts3"]})[list(OUT_COLS.values())]
map_out = OUT_DIR / MAP_OUT_NAME
if map_out.exists() and list(pd.read_csv(map_out, dtype=str).columns) == list(OUT_COLS.values()):
    print("[exists with expected columns, skipped]", map_out.relative_to(ROOT).as_posix())
else:
    pl_map.to_csv(map_out, index=False)
    print("[saved]", map_out.relative_to(ROOT).as_posix(), pl_map.shape)

# 6) Compare with the existing file (row order does not matter)
old = pd.read_csv(MAP_EXISTING, dtype=str)
same_cols = list(old.columns) == list(pl_map.columns)
if same_cols:
    old_s = old.sort_values(OUT_COLS["gmina"]).reset_index(drop=True)
    new_s = pl_map.sort_values(OUT_COLS["gmina"]).reset_index(drop=True)
    identical = len(old_s) == len(new_s) and old_s.equals(new_s)
else:
    identical = False

summary = pd.Series({
    "rows_existing": len(old),
    "rows_rebuilt": len(pl_map),
    "same_columns_in_order": same_cols,
    "identical_after_sort": identical,
}, name="value")
print("AZONOS" if identical else "ELTÉRÉS")
print(summary.to_string())

cmp_path = OUT_DIR / MAP_COMPARE_NAME
summary.to_csv(cmp_path)
print("[saved]", cmp_path.relative_to(ROOT).as_posix())

In [ ]:
# ---- Config ----
CZ_NUTS3_OUT_NAME = "cz_obec_nuts3.csv"
CZ_LAU1_COLS = {"orig": ("op_obec_kod", "op_lau1_kod"),
                "dest": ("doj_obec_kod", "doj_lau1_kod")}
CZ_NUTS3_PREFIX_LEN = 5            # NUTS3 code = first 5 characters of the LAU1 code
CZ_MAP_COLS = ["obec_code", "lau1_code", "nuts3_code"]
MASTER_TABLE = ROOT / "data" / "processed" / "nuts3_master_table.parquet"
MASTER_COLS = {"nuts3": "nuts3_id", "country": "country_code"}
MASTER_COUNTRY = "CZ"
# ----------------

# Read the rebuilt CZ table from disk (checkpoint), both origin and destination codes
cz_tab = pd.read_parquet(cz_out)
parts = []
for side, (obec_col, lau1_col) in CZ_LAU1_COLS.items():
    parts.append(cz_tab[[obec_col, lau1_col]].set_axis(CZ_MAP_COLS[:2], axis=1))
cz_map = pd.concat(parts, ignore_index=True).drop_duplicates()

# 1) One LAU1 per obec, whichever column it came from
per_obec = cz_map.groupby("obec_code")["lau1_code"].nunique()
print(f"obec in the table: {len(per_obec)} | with more than one LAU1: {int((per_obec > 1).sum())}")
assert (per_obec == 1).all(), "an obec has several LAU1 codes"

# 2) NUTS3 code from the LAU1 prefix
cz_map["nuts3_code"] = cz_map["lau1_code"].str[:CZ_NUTS3_PREFIX_LEN]
cz_map = cz_map[CZ_MAP_COLS].sort_values("obec_code").reset_index(drop=True)

# 3) Every node of the CZ graph needs a NUTS3 code
nodes = set(graphs["CZ"].nodes)
keys = set(cz_map["obec_code"])
print(f"graph nodes: {len(nodes)} | without NUTS3: {len(nodes - keys)} "
      f"| in the table only: {len(keys - nodes)}")
assert not (nodes - keys), "graph nodes without a NUTS3 code"

# 4) NUTS3 codes must be the CZ regions of the master table
print("NUTS3 regions from the prefix:", cz_map["nuts3_code"].nunique())
if MASTER_TABLE.exists():
    mt = pd.read_parquet(MASTER_TABLE, columns=list(MASTER_COLS.values()))
    master_ids = set(mt.loc[mt[MASTER_COLS["country"]] == MASTER_COUNTRY, MASTER_COLS["nuts3"]])
    only_map = sorted(set(cz_map["nuts3_code"]) - master_ids)
    only_master = sorted(master_ids - set(cz_map["nuts3_code"]))
    print(f"master table CZ regions: {len(master_ids)} | only in the map: {only_map} "
          f"| only in the master table: {only_master}")
else:
    print("WARNING: master table missing, comparison skipped:",
          MASTER_TABLE.relative_to(ROOT).as_posix())

# 5) Save (skip if it already exists with the expected columns)
cz_map_out = OUT_DIR / CZ_NUTS3_OUT_NAME
if cz_map_out.exists() and list(pd.read_csv(cz_map_out, dtype=str).columns) == CZ_MAP_COLS:
    print("[exists with expected columns, skipped]", cz_map_out.relative_to(ROOT).as_posix())
else:
    cz_map.to_csv(cz_map_out, index=False)
    print("[saved]", cz_map_out.relative_to(ROOT).as_posix(), cz_map.shape)

In [ ]:
# ---- Config ----
METHODS = ["leiden", "louvain", "cnm"]
CMP_NAME = "community_vs_nuts3.csv"
ASSIGN_NAMES = {"CZ": "cz_node_assignment.csv", "PL": "pl_node_assignment.csv"}
# settlement -> NUTS3 tables built above (CZ: cz_map_out, PL: map_out)
MAPS = {
    "CZ": {"path": cz_map_out, "node": "obec_code", "nuts3": "nuts3_code"},
    "PL": {"path": map_out, "node": OUT_COLS["gmina"], "nuts3": OUT_COLS["nuts3"]},
}
# ----------------

def inside_share(edges, labels):
    """Share of the total edge weight whose two ends carry the same label."""
    same = edges["node_a"].map(labels) == edges["node_b"].map(labels)
    return float(edges.loc[same, "weight"].sum() / edges["weight"].sum())

rows = []
for level, G in graphs.items():
    nodes = list(G.nodes)
    edges = pd.read_parquet(OUT_DIR / LEVELS[level]["edges_name"])

    # Settlement -> NUTS3 key, aligned with the graph nodes
    mp = pd.read_csv(MAPS[level]["path"], dtype=str)
    nuts_of = mp.drop_duplicates(MAPS[level]["node"]).set_index(MAPS[level]["node"])[MAPS[level]["nuts3"]]
    assert set(nodes) <= set(nuts_of.index), f"{level}: graph nodes without NUTS3"
    assign = pd.DataFrame({"node": nodes}).set_index("node")
    assign["nuts3"] = nuts_of.reindex(nodes).values

    # NUTS3 partition: modularity and inside share
    nuts_groups = [set(g.index) for _, g in assign.groupby("nuts3")]
    q_nuts3 = nx.community.modularity(G, nuts_groups, weight=WEIGHT_ATTR)
    share_nuts3 = inside_share(edges, assign["nuts3"])
    nuts_codes, _ = pd.factorize(assign["nuts3"])

    for method in METHODS:
        mem = pd.read_parquet(OUT_DIR / f"{level.lower()}_communities_{method}.parquet")
        assign[method] = mem.set_index("node")["community"].reindex(nodes).values
        assert assign[method].notna().all(), f"{level} {method}: node without community"

        comm = assign[method].astype(int)
        groups = [set(g.index) for _, g in assign.groupby(method)]
        rows.append({
            "level": level,
            "method": method,
            "communities": len(groups),
            "nuts3_regions": len(nuts_groups),
            "modularity_communities": round(nx.community.modularity(G, groups, weight=WEIGHT_ATTR), 4),
            "modularity_nuts3": round(q_nuts3, 4),
            "inside_share_communities": round(inside_share(edges, comm), 4),
            "inside_share_nuts3": round(share_nuts3, 4),
            "nmi": round(ig.compare_communities(list(comm), list(nuts_codes), method="nmi"), 4),
            "ari": round(ig.compare_communities(list(comm), list(nuts_codes), method="adjusted_rand"), 4),
        })

    # Save the node-level assignment of all methods (skip if it already exists)
    assign_path = OUT_DIR / ASSIGN_NAMES[level]
    if assign_path.exists():
        print("[exists, skipped]", assign_path.relative_to(ROOT).as_posix())
    else:
        assign.reset_index().to_csv(assign_path, index=False)
        print("[saved]", assign_path.relative_to(ROOT).as_posix(), assign.shape)

cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))

cmp_path = OUT_DIR / CMP_NAME
cmp_df.to_csv(cmp_path, index=False)
print("[saved]", cmp_path.relative_to(ROOT).as_posix())

In [ ]:
import contextlib
import json
import os
import sys

import numpy as np

# ---- Config ----
CZ_GEO = ROOT / "data" / "interim" / "cz_lau2021_3035.geojson"
LAU_GEO = ROOT / "data" / "raw" / "gisco" / "LAU_RG_01M_2021_3035.geojson"
NUTS3_GEO = ROOT / "data" / "raw" / "nuts3" / "NUTS_RG_03M_2024_4326.geojson"
PL_GEO_NAME = "pl_lau2021_3035.geojson"
CRS_PLOT = "EPSG:3035"
CRS_PLOT_URN = "urn:ogc:def:crs:EPSG::3035"     # written into the saved PL file
CRS_NUTS_SRC = "EPSG:4326"
COUNTRY_COL, LAU_COL, NUTS_ID_COL = "CNTR_CODE", "LAU_ID", "NUTS_ID"
POP_COL_GEO = "POP_2021"                         # GISCO LAU 2021 population
PL_COUNTRY = "PL"
NUTS_COUNTRIES = ["CZ", "PL"]
BOUNDS_MARGIN_M = 50_000          # LAU and NUTS3 bounds may differ by this much (different generalisation)
# ----------------

@contextlib.contextmanager
def quiet_stderr():
    """Silence C-level stderr (library messages can contain absolute paths)."""
    sys.stderr.flush()
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(devnull)

def read_features(path):
    """Read the features of a GeoJSON file with the standard library only."""
    with open(path, encoding="utf-8") as f:
        return json.load(f)["features"]

def to_polygons(geometry):
    """Return a list of polygons; a polygon is a list of rings (N x 2 float arrays)."""
    if geometry["type"] == "Polygon":
        polys = [geometry["coordinates"]]
    elif geometry["type"] == "MultiPolygon":
        polys = geometry["coordinates"]
    else:
        raise ValueError(geometry["type"])
    return [[np.asarray(ring, dtype=float)[:, :2] for ring in poly] for poly in polys]

def bounds_of(*polys_by_key):
    """Overall (xmin, ymin, xmax, ymax) of one or more key -> polygons dictionaries."""
    xy = np.vstack([ring for d in polys_by_key for polys in d.values()
                    for poly in polys for ring in poly])
    return np.array([*xy.min(axis=0), *xy.max(axis=0)])

# 1) CZ settlement polygons and population (the file declares no CRS: EPSG:3035 is assumed, checked in step 5)
cz_feats = read_features(CZ_GEO)
cz_poly = {str(ft["properties"][LAU_COL]): to_polygons(ft["geometry"]) for ft in cz_feats}
cz_pop = {str(ft["properties"][LAU_COL]): ft["properties"][POP_COL_GEO] for ft in cz_feats}
assert len(cz_poly) == len(cz_feats), "duplicate CZ LAU_ID"
del cz_feats

# 2) PL settlement polygons: extract once from the EU file, then reuse the small file
pl_geo_path = OUT_DIR / PL_GEO_NAME
if pl_geo_path.exists():
    pl_feats = read_features(pl_geo_path)
    print("[loaded]", pl_geo_path.relative_to(ROOT).as_posix())
else:
    all_feats = read_features(LAU_GEO)
    pl_feats = [ft for ft in all_feats if ft["properties"][COUNTRY_COL] == PL_COUNTRY]
    del all_feats
    with open(pl_geo_path, "w", encoding="utf-8") as f:
        json.dump({"type": "FeatureCollection",
                   "crs": {"type": "name", "properties": {"name": CRS_PLOT_URN}},
                   "features": pl_feats}, f)
    print("[saved]", pl_geo_path.relative_to(ROOT).as_posix(),
          f"({pl_geo_path.stat().st_size / 1e6:.1f} MB, {len(pl_feats)} features)")

# 3) Join keys: PL 13-digit LAU_ID -> 6-digit gmina key through the rebuilt table
pl_map_tab = pd.read_csv(map_out, dtype=str)
gmina_of_lau = dict(zip(pl_map_tab[OUT_COLS["lau"]], pl_map_tab[OUT_COLS["gmina"]]))
pl_keys = [gmina_of_lau.get(str(ft["properties"][LAU_COL])) for ft in pl_feats]
print(f"PL polygons: {len(pl_feats)} | without gmina key: {sum(k is None for k in pl_keys)}")
assert all(k is not None for k in pl_keys), "PL polygon without gmina key"
assert len(set(pl_keys)) == len(pl_keys), "duplicate PL gmina key"
pl_poly = {k: to_polygons(ft["geometry"]) for k, ft in zip(pl_keys, pl_feats)}
pl_pop = {k: ft["properties"][POP_COL_GEO] for k, ft in zip(pl_keys, pl_feats)}
del pl_feats

for level, poly in {"CZ": cz_poly, "PL": pl_poly}.items():
    nodes_g, keys_g = set(graphs[level].nodes), set(poly)
    print(f"{level}: polygons {len(keys_g)} | graph nodes without polygon: {len(nodes_g - keys_g)} "
          f"| polygons without graph node: {len(keys_g - nodes_g)}")
    assert nodes_g == keys_g, f"{level}: polygons and graph nodes differ"

# Population per settlement: every node needs a value
for level, pop in {"CZ": cz_pop, "PL": pl_pop}.items():
    s = pd.Series(pop, dtype=float)
    print(f"{level}: {POP_COL_GEO} sum {s.sum():,.0f} | missing: {int(s.isna().sum())}")
    assert s.notna().all(), f"{level}: missing {POP_COL_GEO}"

# 4) NUTS3 polygons: 2024 edition vs our 2021-based codes, then reproject to EPSG:3035
try:
    nuts_feats = [ft for ft in read_features(NUTS3_GEO)
                  if ft["properties"][COUNTRY_COL] in NUTS_COUNTRIES]
    with quiet_stderr():
        from pyproj import Transformer
        transformer = Transformer.from_crs(CRS_NUTS_SRC, CRS_PLOT, always_xy=True)
    nuts_poly = {}
    for ft in nuts_feats:
        polys = to_polygons(ft["geometry"])
        nuts_poly[ft["properties"][NUTS_ID_COL]] = [
            [np.column_stack(transformer.transform(r[:, 0], r[:, 1])) for r in poly]
            for poly in polys]
    del nuts_feats
except Exception as e:
    raise RuntimeError(f"NUTS3 read/reproject failed: {type(e).__name__}") from None

codes_ours = set(pd.read_csv(cz_map_out, dtype=str)["nuts3_code"]) | set(pl_map_tab[OUT_COLS["nuts3"]])
codes_geo = set(nuts_poly)
print(f"NUTS3 polygons: {len(codes_geo)} | our codes: {len(codes_ours)} "
      f"| only in polygons: {sorted(codes_geo - codes_ours)} "
      f"| only in our codes: {sorted(codes_ours - codes_geo)}")

# 5) Coordinate check per country: settlement polygons vs the NUTS3 polygons of the same country
#    (the NUTS3 file was reprojected from EPSG:4326, so this also confirms the CZ file is in EPSG:3035)
for level, poly in {"CZ": cz_poly, "PL": pl_poly}.items():
    nuts_country = {c: v for c, v in nuts_poly.items() if c.startswith(level)}
    b_lau, b_nuts = bounds_of(poly), bounds_of(nuts_country)
    diff = np.abs(b_lau - b_nuts).max()
    print(f"{level} bounds LAU : {[round(v) for v in b_lau]}")
    print(f"{level} bounds NUTS: {[round(v) for v in b_nuts]} | max difference {diff:,.0f} m "
          f"| {'OK' if diff < BOUNDS_MARGIN_M else 'MISMATCH'}")
    assert diff < BOUNDS_MARGIN_M, f"{level}: LAU and NUTS3 polygons are not aligned"

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

# ---- Config ----
PLOT_METHOD = "leiden"                      # which community method to draw
FIG_NAME = "fig_communities_vs_nuts3_map.png"
FIG_DIR = ROOT / "figures" / NOTEBOOK       # figures are outputs, not processed data
FIG_DPI = 200
_tab = plt.get_cmap("tab20").colors
PALETTE = _tab[0::2] + _tab[1::2]           # 10 dark colours first, then their light pairs
NUTS_EDGE_COLOR = "black"
NUTS_WIDTH_OVER_COMMUNITIES = 0.9           # left panel: NUTS3 borders over the communities
NUTS_WIDTH_OWN_PANEL = 0.35                 # right panel: thin borders between same-coloured regions
SETTLEMENT_EDGE_WIDTH = 0.1
PAIR_SHARE_MIN = 0.01                       # two groups count as neighbours above this commuting share
ASSIGN_FILES = {"CZ": "cz_node_assignment.csv", "PL": "pl_node_assignment.csv"}
EDGE_FILES = {"CZ": "cz_edges_undirected.parquet", "PL": "pl_edges_undirected.parquet"}
POLY = {"CZ": cz_poly, "PL": pl_poly}       # settlement polygons from the geometry cell
COMPARE_FILE = "community_vs_nuts3.csv"
# ----------------

def ring_area(ring):
    """Absolute polygon area of one ring (shoelace formula)."""
    x, y = ring[:, 0], ring[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def outer_rings(poly_dict, keys=None):
    """Outer rings of all polygon parts, largest first, with the key of each ring."""
    items = [(k, poly[0]) for k, polys in poly_dict.items()
             if keys is None or k in keys for poly in polys]
    items.sort(key=lambda kr: -ring_area(kr[1]))
    return [k for k, _ in items], [r for _, r in items]

def unique_colors(labels):
    """One distinct colour per label, the largest groups get the dark colours."""
    order = labels.value_counts().index
    assert len(order) <= len(PALETTE), "more groups than palette colours"
    return {lab: PALETTE[i] for i, lab in enumerate(order)}

def group_colors(labels, edges):
    """Colour per label: greedy colouring so strongly commuting groups differ."""
    la, lb = edges["node_a"].map(labels), edges["node_b"].map(labels)
    pairs = pd.DataFrame({"a": la, "b": lb, "w": edges["weight"]})
    # total commuting weight touching each group, to judge a pair relative to the smaller group
    strength = pd.concat([pairs.groupby("a")["w"].sum(), pairs.groupby("b")["w"].sum()]) \
        .groupby(level=0).sum()
    cross = pairs[pairs["a"] != pairs["b"]].groupby(["a", "b"], as_index=False)["w"].sum()
    Q = nx.Graph()
    Q.add_nodes_from(labels.unique())
    for a, b, w in cross.itertuples(index=False, name=None):
        if w / min(strength[a], strength[b]) >= PAIR_SHARE_MIN:
            Q.add_edge(a, b)
    coloring = nx.greedy_color(Q, strategy="largest_first")
    return {lab: PALETTE[c % len(PALETTE)] for lab, c in coloring.items()}

def draw(ax, poly_dict, node_color, nuts_codes, nuts_width):
    """Filled settlements, NUTS3 borders on top."""
    keys, rings = outer_rings(poly_dict)
    fc = [node_color[k] for k in keys]
    ax.add_collection(PolyCollection(rings, facecolors=fc, edgecolors=fc,
                                     linewidths=SETTLEMENT_EDGE_WIDTH))
    _, nrings = outer_rings(nuts_poly, keys=nuts_codes)
    ax.add_collection(PolyCollection(nrings, facecolors="none", edgecolors=NUTS_EDGE_COLOR,
                                     linewidths=nuts_width))
    xy = np.vstack(rings)
    ax.set_xlim(xy[:, 0].min(), xy[:, 0].max())
    ax.set_ylim(xy[:, 1].min(), xy[:, 1].max())
    ax.set_aspect("equal")
    ax.axis("off")

cmp_tab = pd.read_csv(OUT_DIR / COMPARE_FILE)
fig, axes = plt.subplots(2, 2, figsize=(14, 13), gridspec_kw={"height_ratios": [0.62, 1]})

for row, level in enumerate(["CZ", "PL"]):
    assign = pd.read_csv(OUT_DIR / ASSIGN_FILES[level], dtype={"node": str, "nuts3": str}).set_index("node")
    edges = pd.read_parquet(OUT_DIR / EDGE_FILES[level])
    nuts_codes = {c for c in nuts_poly if c.startswith(level)}
    info = cmp_tab[(cmp_tab["level"] == level) & (cmp_tab["method"] == PLOT_METHOD)].iloc[0]

    # Left: one distinct colour per community; right: greedy colours + thin NUTS3 borders
    comm_col = unique_colors(assign[PLOT_METHOD])
    nuts_col = group_colors(assign["nuts3"], edges)

    draw(axes[row, 0], POLY[level], {n: comm_col[c] for n, c in assign[PLOT_METHOD].items()},
         nuts_codes, NUTS_WIDTH_OVER_COMMUNITIES)
    axes[row, 0].set_title(f"{level}: {PLOT_METHOD} communities (n={int(info['communities'])}, "
                           f"Q={info['modularity_communities']:.3f}) + NUTS3 borders")
    draw(axes[row, 1], POLY[level], {n: nuts_col[c] for n, c in assign["nuts3"].items()},
         nuts_codes, NUTS_WIDTH_OWN_PANEL)
    axes[row, 1].set_title(f"{level}: NUTS3 regions (n={int(info['nuts3_regions'])}, "
                           f"Q={info['modularity_nuts3']:.3f})")

fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_NAME
fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
print("[saved]", fig_path.relative_to(ROOT).as_posix())
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ---- Config ----
PLOT_METHOD = "leiden"
POP = {"CZ": cz_pop, "PL": pl_pop}               # settlement population (POP_2021) from the geometry cell
ASSIGN_FILES = {"CZ": "cz_node_assignment.csv", "PL": "pl_node_assignment.csv"}
OVERLAP_NAMES = {"CZ": "cz_community_nuts3_overlap.csv", "PL": "pl_community_nuts3_overlap.csv"}
PURITY_NAME = "community_nuts3_purity.csv"
HEAT_FIG_NAME = "fig_community_nuts3_heatmap.png"
FIG_DIR = ROOT / "figures" / NOTEBOOK
FIG_DPI = 200
CMAP = "viridis"
TICK_FONT = {"CZ": 9, "PL": 5}
# ----------------

levels = ["CZ", "PL"]
shares, purity_rows = {}, []

for level in levels:
    assign = pd.read_csv(OUT_DIR / ASSIGN_FILES[level],
                         dtype={"node": str, "nuts3": str}).set_index("node")

    pop = pd.Series(POP[level], dtype=float).reindex(assign.index)
    assert pop.notna().all(), f"{level}: nodes without population"
    df = pd.DataFrame({"comm": assign[PLOT_METHOD].values,
                       "nuts3": assign["nuts3"].values,
                       "pop": pop.values})

    # Population in each (community, NUTS3) cell
    tab = df.pivot_table(index="comm", columns="nuts3", values="pop",
                         aggfunc="sum", fill_value=0)

    # Rows: communities by size; columns: grouped by their dominant community
    tab = tab.loc[tab.sum(axis=1).sort_values(ascending=False).index]
    share = tab / tab.sum(axis=0)                      # each NUTS3 column sums to 1
    row_pos = {c: i for i, c in enumerate(share.index)}
    dominant = share.idxmax(axis=0).map(row_pos)
    col_order = pd.DataFrame({"dom": dominant, "top": -share.max(axis=0)}) \
        .sort_values(["dom", "top"]).index
    tab, share = tab[col_order], share[col_order]
    shares[level] = share

    # Purity (population-weighted mean of the largest share):
    #   NUTS3 side     = sum of the largest cell of each column / total population
    #   community side = sum of the largest cell of each row / total population
    total_pop = tab.values.sum()
    purity_rows.append({"level": level, "method": PLOT_METHOD,
                        "nuts3_regions": tab.shape[1], "communities": tab.shape[0],
                        "nuts3_purity": round(float(tab.max(axis=0).sum() / total_pop), 4),
                        "community_purity": round(float(tab.max(axis=1).sum() / total_pop), 4)})

    out_path = OUT_DIR / OVERLAP_NAMES[level]
    tab.to_csv(out_path)
    print("[saved]", out_path.relative_to(ROOT).as_posix(), tab.shape)

# Save the small purity table so it can be read without re-running the notebook
purity_df = pd.DataFrame(purity_rows)
print(purity_df.to_string(index=False))
purity_path = OUT_DIR / PURITY_NAME
purity_df.to_csv(purity_path, index=False)
print("[saved]", purity_path.relative_to(ROOT).as_posix())

# Heatmap: one panel per country, heights proportional to the number of communities
heights = [shares[lv].shape[0] for lv in levels]
fig, axes = plt.subplots(2, 1, figsize=(16, 13), constrained_layout=True,
                         gridspec_kw={"height_ratios": heights})
for ax, level in zip(axes, levels):
    sh = shares[level]
    im = ax.imshow(sh.values, aspect="auto", cmap=CMAP, vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks(range(sh.shape[1]))
    ax.set_xticklabels(sh.columns, rotation=90, fontsize=TICK_FONT[level])
    ax.set_yticks(range(sh.shape[0]))
    ax.set_yticklabels(sh.index, fontsize=8)
    ax.set_xlabel("NUTS3 region")
    ax.set_ylabel(f"{PLOT_METHOD} community")
    ax.set_title(f"{level}: share of each NUTS3 region's population by community")
fig.colorbar(im, ax=axes, shrink=0.6, label="share of the NUTS3 region's population")

FIG_DIR.mkdir(parents=True, exist_ok=True)
heat_path = FIG_DIR / HEAT_FIG_NAME
fig.savefig(heat_path, dpi=FIG_DPI, bbox_inches="tight")
print("[saved]", heat_path.relative_to(ROOT).as_posix())
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection

# ---- Config ----
PLOT_METHOD = "leiden"
GRAPH_FIG_NAME = "fig_community_graph.png"
FLOWS_NAMES = {"CZ": "cz_community_flows.csv", "PL": "pl_community_flows.csv"}
NODE_AREA_MAX = 2600            # marker area (points^2) of the most populous community
EDGE_WIDTH_MAX = 14             # line width (points) of the strongest link
EDGE_WIDTH_EXPONENT = 0.5       # width ~ flow ** 0.5, so weak links stay visible
EDGE_ALPHA = 0.45
NUTS_LINE_COLOR, NUTS_LINE_WIDTH = "0.6", 0.5
LABEL_FONT = 9
TOP_PAIRS = 3
# ----------------

def centroid_of_largest_part(polys):
    """Centroid of the largest part's outer ring (shoelace formula)."""
    best = max(polys, key=lambda poly: ring_area(poly[0]))[0]
    x, y = best[:, 0], best[:, 1]
    cross = x * np.roll(y, -1) - np.roll(x, -1) * y
    a2 = cross.sum()
    if a2 == 0:                                   # degenerate ring, use the vertex mean
        return float(x.mean()), float(y.mean())
    cx = ((x + np.roll(x, -1)) * cross).sum() / (3 * a2)
    cy = ((y + np.roll(y, -1)) * cross).sum() / (3 * a2)
    return float(cx), float(cy)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, level in zip(axes, ["CZ", "PL"]):
    assign = pd.read_csv(OUT_DIR / ASSIGN_FILES[level],
                         dtype={"node": str, "nuts3": str}).set_index("node")
    edges = pd.read_parquet(OUT_DIR / EDGE_FILES[level])
    comm_of = assign[PLOT_METHOD]

    # Settlement centroids and population -> community position and size
    cent = pd.DataFrame({n: centroid_of_largest_part(POLY[level][n]) for n in comm_of.index},
                        index=["x", "y"]).T
    pop = pd.Series(POP[level], dtype=float).reindex(comm_of.index)
    cent["pop"] = pop
    cent["comm"] = comm_of
    cent["wx"], cent["wy"] = cent["x"] * cent["pop"], cent["y"] * cent["pop"]
    grp = cent.groupby("comm")[["wx", "wy", "pop"]].sum()
    pos = pd.DataFrame({"x": grp["wx"] / grp["pop"], "y": grp["wy"] / grp["pop"], "pop": grp["pop"]})

    # Commuting between communities: sum of the edge weights over each community pair
    la, lb = edges["node_a"].map(comm_of), edges["node_b"].map(comm_of)
    total_w = float(edges["weight"].sum())
    inside_w = float(edges.loc[la == lb, "weight"].sum())
    pairs = (pd.DataFrame({"a": np.minimum(la, lb), "b": np.maximum(la, lb), "w": edges["weight"]})
             .query("a != b").groupby(["a", "b"], as_index=False)["w"].sum()
             .rename(columns={"a": "comm_a", "b": "comm_b", "w": "weight"}))
    pairs["share_of_total"] = pairs["weight"] / total_w
    pairs = pairs.sort_values("weight", ascending=False).reset_index(drop=True)

    flows_path = OUT_DIR / FLOWS_NAMES[level]
    pairs.to_csv(flows_path, index=False)
    print(f"[saved] {flows_path.relative_to(ROOT).as_posix()} ({len(pairs)} community pairs)")
    print(f"{level}: inside-community share of all commuting {inside_w / total_w:.3f} | "
          f"top {TOP_PAIRS} pairs:",
          [(int(r.comm_a), int(r.comm_b), round(r.share_of_total, 4))
           for r in pairs.head(TOP_PAIRS).itertuples()])

    # Background: thin NUTS3 outlines
    _, nrings = outer_rings(nuts_poly, keys={c for c in nuts_poly if c.startswith(level)})
    ax.add_collection(PolyCollection(nrings, facecolors="none", edgecolors=NUTS_LINE_COLOR,
                                     linewidths=NUTS_LINE_WIDTH, zorder=1))

    # Links: width ~ flow ** exponent
    wmax = pairs["weight"].max()
    segs = [[(pos.loc[r.comm_a, "x"], pos.loc[r.comm_a, "y"]),
             (pos.loc[r.comm_b, "x"], pos.loc[r.comm_b, "y"])] for r in pairs.itertuples()]
    widths = EDGE_WIDTH_MAX * (pairs["weight"] / wmax) ** EDGE_WIDTH_EXPONENT
    ax.add_collection(LineCollection(segs, linewidths=widths, colors="0.2",
                                     alpha=EDGE_ALPHA, zorder=2, capstyle="round"))

    # Nodes: area ~ population, same colours as the map (cell 29)
    col = unique_colors(comm_of)
    sizes = NODE_AREA_MAX * pos["pop"] / pos["pop"].max()
    ax.scatter(pos["x"], pos["y"], s=sizes, c=[col[c] for c in pos.index],
               edgecolors="black", linewidths=0.8, zorder=3)
    for c, r in pos.iterrows():
        ax.text(r["x"], r["y"], str(c), ha="center", va="center", fontsize=LABEL_FONT, zorder=4)

    allxy = np.vstack(nrings)
    ax.set_xlim(allxy[:, 0].min(), allxy[:, 0].max())
    ax.set_ylim(allxy[:, 1].min(), allxy[:, 1].max())
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"{level}: {PLOT_METHOD} communities as nodes "
                 f"(inside share {inside_w / total_w:.2f})")

fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
graph_fig_path = FIG_DIR / GRAPH_FIG_NAME
fig.savefig(graph_fig_path, dpi=FIG_DPI, bbox_inches="tight")
print("[saved]", graph_fig_path.relative_to(ROOT).as_posix())
plt.show()